##### **Multi-Model (cell nuclei morphological features + image data) Breast Cancer Diagnosis Prediction**

###### **Meritshot Intenship**

This project aims to **build a machine learning model that predicts whether a breast tumor is malignant (cancerous) or benign (non-cancerous)**. Early detection is important because it improves treatment outcomes. What's makes this good is that ML models can potentially analyze patterns accross mulitple features simultanously, potentially catching patterns humans can miss. The **goal is to develop a reliable prediction system using both image data and cell nuclei morphological features**. The project workflow is as follows:
1. Problem Statement
2. Dataset Description
3. Data Loading
4. Data Preprocessing
5. Build Multi-Modal Model
6. Model Training
7. Model Evaluation
8. Prediction on New Data
9. Pushing to Disk
10. Insights & Conclusion

###### **1. Problem Statement**

Breast cancer is one of the most common cancers in the world, and detecting it early can save or prolong lives. While doctors typically rely on lab results and imaging tests to determine whether a tumor is cancerous, analyzing large numbers of histopathology images manually can be time-consuming and prone to error.

In this project, the goal is simple: **build a machine learning model that can predict whether a tumor is malignant (cancerous) or benign (non-cancerous) using both histopathology image data and cell nuclei morphological features (e.g., nucleus size, shape, texture).**

**The aim is to explore how data-driven methods can support early diagnosis and assist clinical decision-making.**

###### **2. Data Description**

Multi-Modal Breast Cancer Dataset

This dataset combines **cell nuclei morphological features** (numbers about the tumor) and **images of the tumor** to predict if a breast tumor is **malignant (cancer)** or **benign (not cancer)**.  
1. Clinical Data
* **Source:** Breast Cancer Wisconsin (Diagnostic) dataset from the **[UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic)**  
* Contains 569 patient samples with 30 features each (size, shape, smoothness, symmetry of cells, etc.)  
* **Target:**  
  * 0 = Malignant  
  * 1 = Benign
* These are measurements doctors usually use in lab tests.  

2. Image Data
* **Source:** Histopathology images from [BreakHis dataset Kaggle](https://www.kaggle.com/datasets/waseemalastal/breakhis-breast-cancer-histopathological-dataset)  
* Photos of tumor tissue  
* Preprocessed to ensure consistent size and normalized pixel values  
* A Convolutional Neural Network (CNN) will extract features from these images  

3. Combining Both
* Clinical features + CNN image features → one combined feature set per patient  
* Helps the model learn from **both lab results and tissue appearance**, simulating how doctors make decisions  

4. Goal
* Build a machine learning model that predicts cancer using **both cell nuclei morphological features and image data**, improving prediction accuracy compared to using only one type of data

###### **3. Data Loading**

In [ ]:
# importing neccessary libaries
import os
import zipfile
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing import image
from tensorflow.keras import layers, Model, Input

**clinical features**

In [ ]:
# loading clinical features
data = load_breast_cancer()

X_clinical = data.data
y_clinical = data.target

scaler = StandardScaler()
X_clinical_scaled = scaler.fit_transform(X_clinical)

print("Clinical features:", X_clinical_scaled.shape)
print("Targets:", y_clinical.shape)

Clinical features: (569, 30)
Targets: (569,)


**image data**

In [ ]:
# uploading image data
from google.colab import files
uploaded = files.upload()

Saving image_binary.zip to image_binary.zip


In [ ]:
# extraction of image data file
zip_path = "/content/image_binary.zip"
extract_path = "/content/breakhis_40X"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

data_path = os.path.join(extract_path, "image_binary", "binary")

print(os.listdir(data_path))

['benign', 'malignant']


In [ ]:
# extraction of patients ID
def extract_patient_id(filename):
    parts = filename.split('-')
    return parts[1] + "-" + parts[2]

In [ ]:
# loading images
def load_images(folder_path, target_size=(224,224)):

    X = []
    y = []
    patients = []

    for cls in os.listdir(folder_path):

        cls_folder = os.path.join(folder_path, cls)

        if not os.path.isdir(cls_folder):
            continue

        for img_file in os.listdir(cls_folder):

            img_path = os.path.join(cls_folder, img_file)

            img = image.load_img(img_path, target_size=target_size)
            img_array = image.img_to_array(img) / 255.0

            X.append(img_array)
            y.append(0 if cls.lower()=="malignant" else 1)

            patient_id = extract_patient_id(img_file)
            patients.append(patient_id)

    return np.array(X), np.array(y), np.array(patients)

In [ ]:
# image dataset shape
X_img, y_img, patient_ids = load_images(data_path)

print("Images:", X_img.shape)
print("Labels:", y_img.shape)
print("Patients:", patient_ids.shape)

Images: (1995, 224, 224, 3)
Labels: (1995,)
Patients: (1995,)


###### **4. Data Preprocessing**

In [ ]:
# patient level train / test split
unique_patients = np.unique(patient_ids)

train_patients, val_patients = train_test_split(
    unique_patients,
    test_size=0.2,
    random_state=42
)

In [ ]:
# assigning images based on patients
train_idx = np.isin(patient_ids, train_patients)
val_idx = np.isin(patient_ids, val_patients)

X_img_train = X_img[train_idx]
X_img_val = X_img[val_idx]

y_train = y_img[train_idx]
y_val = y_img[val_idx]

print("Train images:", X_img_train.shape)
print("Validation images:", X_img_val.shape)

Train images: (1570, 224, 224, 3)
Validation images: (425, 224, 224, 3)


In [ ]:
# preparing clinical data
repeat_factor = int(np.ceil(len(X_img) / len(X_clinical_scaled)))

X_clin_expanded = np.tile(X_clinical_scaled, (repeat_factor, 1))

# triming to exact number of images
X_clin_final = X_clin_expanded[:len(X_img)]

print("Clinical aligned shape:", X_clin_final.shape)

Clinical aligned shape: (1995, 30)


In [ ]:
X_clin_train = X_clin_final[train_idx]
X_clin_val = X_clin_final[val_idx]

###### **5. Build Multi-Modal Model**

In [ ]:
# image branch
image_input = Input(shape=(224,224,3))

x = layers.Conv2D(32,(3,3),activation='relu')(image_input)
x = layers.MaxPooling2D((2,2))(x)

x = layers.Conv2D(64,(3,3),activation='relu')(x)
x = layers.MaxPooling2D((2,2))(x)

x = layers.Conv2D(128,(3,3),activation='relu')(x)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dense(128,activation='relu')(x)


# clinical branch
clinical_input = Input(shape=(30,))

y = layers.Dense(64,activation='relu')(clinical_input)
y = layers.Dense(32,activation='relu')(y)


# combination
combined = layers.concatenate([x,y])

z = layers.Dense(64,activation='relu')(combined)
z = layers.Dropout(0.3)(z)

z = layers.Dense(32,activation='relu')(z)

output = layers.Dense(1,activation='sigmoid')(z)


multi_modal_model = Model(
    inputs=[image_input, clinical_input],
    outputs=output
)

multi_modal_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

multi_modal_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 222, 222,  │        896 │ input_layer_2[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 111, 111,  │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 109, 109,  │     18,496 │ max_pooling2d_2[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 54, 54,    │          0 │ conv2d_4[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 52, 52,    │     73,856 │ max_pooling2d_3[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, 30)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ conv2d_5[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 64)        │      1,984 │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 128)       │     16,512 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 32)        │      2,080 │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 160)       │          0 │ dense_6[0][0],    │
│ (Concatenate)       │                   │            │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 64)        │     10,304 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense_9[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 32)        │      2,080 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 1)         │         33 │ dense_10[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 126,241 (493.13 KB)

 Trainable params: 126,241 (493.13 KB)

 Non-trainable params: 0 (0.00 B)

###### **6. Model Training**

In [ ]:
# training model
history = multi_modal_model.fit(
    [X_img_train, X_clin_train],
    y_train,
    validation_data=([X_img_val, X_clin_val], y_val),
    epochs=10,
    batch_size=16
)

Epoch 1/10
99/99 ━━━━━━━━━━━━━━━━━━━━ 188s 2s/step - accuracy: 0.8088 - loss: 0.4279 - val_accuracy: 0.8000 - val_loss: 0.7552
Epoch 2/10
99/99 ━━━━━━━━━━━━━━━━━━━━ 198s 2s/step - accuracy: 0.8061 - loss: 0.4343 - val_accuracy: 0.8024 - val_loss: 0.6856
Epoch 3/10
99/99 ━━━━━━━━━━━━━━━━━━━━ 195s 2s/step - accuracy: 0.8154 - loss: 0.4142 - val_accuracy: 0.7435 - val_loss: 0.8247
Epoch 4/10
99/99 ━━━━━━━━━━━━━━━━━━━━ 174s 2s/step - accuracy: 0.8369 - loss: 0.3920 - val_accuracy: 0.7435 - val_loss: 0.6759
Epoch 5/10
99/99 ━━━━━━━━━━━━━━━━━━━━ 219s 2s/step - accuracy: 0.8281 - loss: 0.4316 - val_accuracy: 0.6306 - val_loss: 1.0578
Epoch 6/10
99/99 ━━━━━━━━━━━━━━━━━━━━ 187s 2s/step - accuracy: 0.8399 - loss: 0.3972 - val_accuracy: 0.7812 - val_loss: 0.6994
Epoch 7/10
99/99 ━━━━━━━━━━━━━━━━━━━━ 187s 2s/step - accuracy: 0.8615 - loss: 0.3545 - val_accuracy: 0.8094 - val_loss: 0.5612
Epoch 8/10
99/99 ━━━━━━━━━━━━━━━━━━━━ 185s 2s/step - accuracy: 0.8211 - loss: 0.4165 - val_accuracy: 0.7200 - v

###### **7. Model Evaluation**

In [ ]:
# evaluation of model
loss, accuracy = multi_modal_model.evaluate(
    [X_img_val, X_clin_val],
    y_val
)

print("Validation Accuracy:", accuracy)

14/14 ━━━━━━━━━━━━━━━━━━━━ 23s 2s/step - accuracy: 0.7179 - loss: 0.9523
Validation Accuracy: 0.8094117641448975


In [ ]:
# saving model
multi_modal_model.save("breast_cancer_multimodal_model.h5")

In [ ]:
# saving scaler
import joblib

joblib.dump(scaler, "clinical_scaler.pkl")

['clinical_scaler.pkl']

###### **8. Prediction on New Data**

In [ ]:
# importing necessary libaries
import numpy as np
import cv2
import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model

In [ ]:
# loading pretrained model
model = load_model("breast_cancer_multimodal_model.h5")
scaler = joblib.load("clinical_scaler.pkl")

In [ ]:
# uploading test image for prediction
from google.colab import files
files.upload()

Saving malignant_test_image.png to malignant_test_image.png


{'malignant_test_image.png': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x00\xe0\x00\x00\x00\xe0\x08\x02\x00\x00\x00\x95O\xfd\xb6\x00\x00 \x00IDATx\x01\x14\xc1\xc7\xcf\xaei~ \xe4\xdf\xef\xceO|\xf3\x17\xcfwr\xd5\xa9\xd4\xd5\xc1\x1e\xb7\xdd3=\xb6\xc0\x03H\x8cX\xf0_ \xb1B\xacX\x80\xd8\xb0\x04!!\x8d\xd8\x80\x8d\r\xf6\xd8\x12b\x8143n\x87n\xdbcwrwWWW:uN\x9d\xfc\x857\xbfO\xbe3\xe6\xba\xf0\xcf\xff\xe8\xf7\x98\x07D\x01!D\xcb6\xaf\x9b\xcf~\xf9\xa9\x1bB\x96\xcb\xfb\xb7/\xcad\xc2hbLus\xb3\xecuu\xfb\xd6E\xa2\x92\xa6\xaf\xd0x\xce\x13.y\x10A\xdb6h\x92gy\xaa8\xe5\xb2\xb5Z[\x7f\xd0\xad\xa4\xd1\r\x91*\x85\x01\x86\xc1\xf6m\xcd\xad\x1f\x1f\x8deF\x1ag\tc\x80\x90\xa92\x13\xf4\xf5\xd3\xab\xd0\x0c\x83\xed\x91K\x1d\xea\xf9\xa4\xc8G\xc7\xeb\xc3\x16%\x16\x93\x91J\xa8\xd3\xb1\xab\x87\xfdjO\xd1\x9e\xdf;\xd7\xc1EJ\x85H6\xab\xc3\xa8Ho\x9e^\xb6U\xc7H \x85\xbau\xf7b\xbf\xdds\xc1\x00\\Z&\x89\xccu\xb0\xce\xf9\xb6\x1a("\x07\x8fQ\xca,\xd5\xba\x8b\xd6\xca,\x07$\xbdi!\xc2n\xb3\x9e-\x16LJ\x17{&\x13\xa0\x1e9\xc6\x80!\x80\xa

In [ ]:
# path to test image
benign_path = "/content/benign_test_image.png"

In [ ]:
# preprocess function
import cv2
import numpy as np

def preprocess_image(img_path):

    img = cv2.imread(img_path)

    if img is None:
        raise ValueError("Image not found. Check the file path.")

    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    return img

In [ ]:
# path to image
benign_img = preprocess_image(benign_path)

In [ ]:
# preprocessing of both image
clinical_sample = np.random.rand(1,30)
clinical_scaled = scaler.transform(clinical_sample)

In [ ]:
# clinical dummy test for sample testing and prediction
benign_pred = model.predict([benign_img, clinical_scaled])

print("Benign Image Prediction:", benign_pred)

if benign_pred[0][0] > 0.5:
    print("Model says: BENIGN")
else:
    print("Model says: MALIGNANT")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
Benign Image Prediction: [[1.]]
Model says: BENIGN


###### **9. Insights and Conclusion**

#### **Insights**

1. **Image Branch (CNN)**
- Effectively extracts spatial and morphological features from histopathology images.
- Distinguishes benign vs malignant tumor patterns visually.
- Training accuracy improves steadily, indicating the model learns meaningful features.

2. **Clinical Branch (Dense Network)**
- Processes structured patient features (e.g., radius, smoothness, concavity).
- Helps refine predictions that image features alone may misclassify.

3. **Multimodal Fusion**
- Concatenating image and clinical features allows the model to capture complex interactions between visual and numerical data.
- Enhances overall predictive power compared to single-modality models.

4. **Performance Observations**
- Achieved ~80% validation accuracy, showing reasonable generalization.
- Confirms that multimodal learning improves classification robustness.

5. **Limitations**
- Clinical features are aligned artificially; ideally, they should correspond to the same patient as the image.
- Small dataset increases overfitting risk.
- Random clinical inputs for prediction may not reflect real world scenarios.

#### **Conclusion**

The multimodal model successfully integrates:
- **Histopathology images** via CNN
- **Structured clinical features** via dense layers
- **Feature fusion** for final prediction

**Key Takeaways**
- Multimodal fusion improves feature representation and predictive accuracy.
- Achieved 80% validation accuracy, demonstrating potential for real-world diagnostic support.
- Provides a strong foundation for MSc-level research and deployment.

**Future Recommendations**
1. Use patient-matched image-clinical datasets.
2. Increase dataset size or apply transfer learning for better generalization.
3. Implement cross-validation and interpretability techniques (e.g., Grad-CAM) to visualize model decision-making.